In [1]:
import pandas as pd
import sys
from pathlib import Path 
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import numpy as np
import geopandas as gpd

sys.path.insert(0, str(Path("../src").resolve()))
from idx_exchange_data_science.preprocessing import initial_cleaning, clean_living_area, select_training_window,\
    filter_close_price, clean_lot_size, clean_coordinates, CoordinateImputer, add_property_age_at_sale, add_bed_bath_ratio,\
    clean_property_characteristics, clean_postal_code, add_binary_missing_indicators, clean_categorical_features,\
    add_school_district_features
    
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV

## Advanced Models: XGBoost and LightGBM

This notebook develops and evaluates XGBoost and LightGBM models to predict residential property closing prices (ClosePrice) using historical California real estate data. The models are evaluated against previously developed baseline models, including Linear Regression, Decision Tree, and Random Forest. Light hyperparameter tuning is performed to optimize model performance.

### Data Loading and Preprocessing

The dataset consists of monthly CSV files from the California Regional Multiple Listing Service (CRMLS), covering January 2024 through April 2026.

The data is split chronologically into training, validation (March 2026), and test (April 2026) sets. Reusable cleaning functions from the idx_exchange.preprocessing_data_science module are applied to prepare the datasets for modeling.

Preprocessing parameters are learned from the training data, with fold-specific fitting used during cross-validation to prevent data leakage.

In [2]:
# Create a Path object to the folder containing the monthly CSV filess
data_path = Path("../data")

# Get all the CSV files from the data folder and sort them chronologically by filename
csv_files = sorted(data_path.glob("*.csv"))

In [3]:
# Split the monthly CSV files into training, validation, and test sets
train_files = csv_files[:-2]
validation_file = csv_files[-2]
test_file = csv_files[-1]

In [4]:
# Read each monthly training file into a DataFrame
train_dfs = []

for file in train_files:
    monthly_data = pd.read_csv(file)

    # Extract the year and month from the CSV filename
    source_month = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    monthly_data["SourceMonth"] = source_month[:4] + "-" + source_month[4:6]
    
    train_dfs.append(monthly_data)
    
# Combine the monthly training DataFrames into a single DataFrame
initial_data = pd.concat(train_dfs, ignore_index=True)

/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_1519/84281947.py:5: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)
/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_1519/84281947.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)
/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_1519/84281947.py:5: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)


In [5]:
# Load validation and test files
validation_df = pd.read_csv(validation_file)
test_df = pd.read_csv(test_file)

# Add source month to each dataset
for df, file in [(validation_df, validation_file), (test_df, test_file)]:
    source_month = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    df["SourceMonth"] = source_month[:4] + "-" + source_month[4:6]

### Initial Data Cleaning

Apply the initial cleaning rules to remove invalid records, including non-California properties, duplicate transactions, non-single-family residences, inconsistent dates, and invalid closing prices. The same cleaning rules are applied to all three datasets.

In [6]:
# Apply the initial cleaning rules to all datasets
clean_train_df, train_log = initial_cleaning(initial_data)
clean_validation_df, validation_log = initial_cleaning(validation_df)
clean_test_df, test_log = initial_cleaning(test_df)

In [7]:
display(train_log)

,step,Before,Removed,Removed (%),Remaining
0,California filtering,568074,76,0.0134,567998
1,Duplicate removal,567998,2911,0.5125,565087
2,Property-type filtering,565087,280370,49.6154,284717
3,Date consistency,284717,39,0.0137,284678
4,Target cleaning,284678,3,0.0011,284675


### Feature Selection

Select the relevant property features, target variable (ClosePrice), and tracking columns required for preprocessing and modeling. 

Features with 80% or more missing values are excluded from modeling. Based on the full 26 months training dataset, `BasementYN`, `BuildingAreaTotal`, and school-related columns exceed this threshold. 

Note: Detailed geographic layer using school districts will be added in Week 6 (Feature engineering). 

In [8]:
feature_columns = ["AttachedGarageYN", "BathroomsTotalInteger", "BedroomsTotal",
                   "City", "CountyOrParish", "FireplaceYN", "GarageSpaces", "Latitude",
                   "Longitude", "LivingArea","LotSizeSquareFeet", "LotSizeAcres", "NewConstructionYN",
                   "ParkingTotal", "PoolPrivateYN", "PostalCode", "ViewYN", "YearBuilt"]

target_column = ["ClosePrice"]
tracking_columns = ["SourceMonth", "CloseDate"]

key_columns = feature_columns + target_column + tracking_columns

In [9]:
clean_train_df = clean_train_df[key_columns].copy()
clean_validation_df = clean_validation_df[key_columns].copy()
clean_test_df = clean_test_df[key_columns].copy()

In [10]:
# Perserve the datasets before further preprocessing
base_train_df = clean_train_df.copy()
base_validation_df = clean_validation_df.copy()
base_test_df = clean_test_df.copy()

### Training Window Selection and Target Filtering

Select an adjustable training window of X months preceding March 2026. Calculate the 0.5th and 99.5th percentile ClosePrice thresholds from the selected training data and apply the same thresholds to the datasets according to the agreed cleaning rules.

In [11]:
X = 18  # Try 6, 12, 18

# Selcet training window 
clean_train_df = select_training_window(base_train_df, n_months=X, validation_month="2026-03")

print(f"Training window: {X} months")
print(f"Training records: {len(clean_train_df)}")

Training window: 18 months
Training records: 190549


In [12]:
# Calculate thresholds from the selected X-month training window
lower_threshold = clean_train_df["ClosePrice"].quantile(0.005)
upper_threshold = clean_train_df["ClosePrice"].quantile(0.995)

# Remove records outside the training-derived ClosePrice thresholds
clean_train_df, train_price_log = filter_close_price(clean_train_df, lower_threshold, upper_threshold)
clean_validation_df, validation_price_log = filter_close_price(clean_validation_df, lower_threshold, upper_threshold)
clean_test_df, test_price_log = filter_close_price(clean_test_df, lower_threshold, upper_threshold)

In [13]:
# # Apply log transformation to ClosePrice for all datasets
for df in[clean_train_df, clean_validation_df, clean_test_df]:
    df["LogClosePrice"] = np.log1p(df["ClosePrice"])

### Numerical Feature Cleaning

#### LivingArea and LotSizeSquareFeet

Clean numerical property features by removing records with nonpositive LivingArea values while retaining missing values for later imputation. Standardize lot-size measurements by converting valid acreage values to square feet when LotSizeSquareFeet is missing.

In [14]:
# Remove nonpositive LivingArea values while retaining missing values
clean_train_df, train_area_log = clean_living_area(clean_train_df)
clean_validation_df, validation_log = clean_living_area(clean_validation_df)
clean_test_df, test_log = clean_living_area(clean_test_df)

In [15]:
# Clean lot sizes and recover missing square footage using acres
clean_train_df, train_recovered_rows = clean_lot_size(clean_train_df)
clean_validation_df, validation_recovered_rows = clean_lot_size(clean_validation_df)
clean_test_df, test_recovered_rows = clean_lot_size(clean_test_df)

In [16]:
# Summarize the number of recovered lot-size values in each dataset
display(pd.DataFrame({
    "Dataset": ["Training", "Validation", "Test"],
    "Lot sizes recovered": [train_recovered_rows, validation_recovered_rows, test_recovered_rows]}))

,Dataset,Lot sizes recovered
0,Training,15
1,Validation,2
2,Test,0


#### Property Characteristics 

Property characteristics, including bedrooms, bathrooms, garage spaces, and parking counts, are evaluated using provisional plausibility ranges. Invalid values are marked as missing for later imputation rather than removing the corresponding records.

These cleaning rules have not yet been finalized and may be revised following team discussion and further review.

In [17]:
# Clean property characteristics and record invalid values
clean_train_df, train_property_log = clean_property_characteristics(clean_train_df)
clean_validation_df, validation_property_log = clean_property_characteristics(clean_validation_df)
clean_test_df, test_property_log = clean_property_characteristics(clean_test_df)

In [18]:
# Summarize invalid property characteristic values
property_logs = pd.DataFrame({ "Training": train_property_log,
                               "Validation": validation_property_log,
                               "Test": test_property_log})

display(property_logs)

,Training,Validation,Test
BedroomsTotal,61,6,2
BathroomsTotalInteger,44,5,3
GarageSpaces,343,19,21
ParkingTotal,707,45,38


### Geographic Feature Cleaning

Standardize postal codes to five-digit ZIP codes and mark latitude and longitude values outside California's geographic bounds as missing. Missing coordinates will be imputed later using training-derived ZIP-level, county-level, and overall medians within the modeling pipeline.

In [19]:
# Standardize postal codes for all datasets
clean_train_df = clean_postal_code(clean_train_df)
clean_validation_df = clean_postal_code(clean_validation_df)
clean_test_df = clean_postal_code(clean_test_df)

In [20]:
# Mark invalid coordinates as missing
clean_train_df = clean_coordinates(clean_train_df)
clean_validation_df = clean_coordinates(clean_validation_df)
clean_test_df = clean_coordinates(clean_test_df)

### Feature Engineering

Calculate property age and create missingness indicators for binary property features. Missing values are retained for imputation within the modeling pipeline.

In [21]:
# Calculate property age at sale for all datasets
clean_train_df = add_property_age_at_sale(clean_train_df)
clean_validation_df = add_property_age_at_sale(clean_validation_df)
clean_test_df = add_property_age_at_sale(clean_test_df)

In [22]:
# Apply missingness indicator for binary features to all datasets
clean_train_df = add_binary_missing_indicators(clean_train_df)
clean_validation_df = add_binary_missing_indicators(clean_validation_df)
clean_test_df = add_binary_missing_indicators(clean_test_df)

### Categorical Feature Cleaning

Standardize `City`, `CountyOrParish`, and `PostalCode` by converting values to strings, removing surrounding whitespace, and replacing missing values with `"Unknown"`.

Target encoding will be performed within the modeling pipeline. The encoding approach is provisional and subject to further team review.

In [23]:
# Standardize categorical features for target encoding
clean_train_df = clean_categorical_features(clean_train_df)
clean_validation_df = clean_categorical_features(clean_validation_df)
clean_test_df = clean_categorical_features(clean_test_df)

## Preprocessing Pipeline

Define preprocessing steps for numerical, binary, and categorical features using sklearn pipelines. Missing values are imputed using training-derived statistics, numerical features are standardized, and categorical features are target-encoded.

Learned preprocessing parameters within the modeling pipeline are fitted separately for each cross-validation training fold to reduce data leakage. Fixed data-cleaning rules and training-derived ClosePrice thresholds are applied before cross-validation

In [24]:
# define the columns used by each pipeline.
area_columns = ["LivingArea", "LotSizeSquareFeet"]
property_columns = ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal"]
coordinate_columns = ["Latitude", "Longitude"]
age_columns = ["PropertyAgeAtSale"]
binary_columns = ["AttachedGarageYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN", "ViewYN"]
binary_missing_columns = [f"{col}_Missing" for col in binary_columns]
categorical_columns = ["City", "CountyOrParish","PostalCode"]

In [25]:
# Impute, log-transform, and standardize area features
area_pipeline = Pipeline([("imputation", SimpleImputer(strategy="median")),
                          ("log", FunctionTransformer(np.log1p)),
                          ("scaling", StandardScaler())])

# Impute property counts using mode, then standardize
property_pipeline = Pipeline([("imputation", SimpleImputer(strategy="most_frequent")),
                              ("scaling", StandardScaler())])

# Impute and standardize property age
property_age_pipeline = Pipeline([("imputation", SimpleImputer(strategy="median")),
                                  ("scaling", StandardScaler())])

# Standardize coordinates after geographic imputation
coordinate_pipeline = Pipeline([("scaling", StandardScaler())])

# Impute binary features using mode
binary_pipeline = Pipeline([("imputation", SimpleImputer(strategy="most_frequent"))])

# Encode high-cardinality categorical features
categorical_pipeline = Pipeline([("target_encoding", TargetEncoder(target_type="continuous", cv=5,\
                                                                   shuffle=True, random_state=42))])

In [26]:
# Build the ColumnTransformer by assigning each pipeline to its corresponding features.
preprocessor = ColumnTransformer([("area", area_pipeline, area_columns),
                                  ("property", property_pipeline, property_columns),
                                  ("coordinates", coordinate_pipeline, coordinate_columns),
                                  ("property_age", property_age_pipeline, age_columns),
                                  ("binary", binary_pipeline, binary_columns),
                                  ("binary_missing", "passthrough", binary_missing_columns),
                                  ("categorical", categorical_pipeline, categorical_columns)])

In [27]:
# Prepare features and target 
model_features = (area_columns + property_columns + coordinate_columns +\
                  age_columns + binary_columns + binary_missing_columns + categorical_columns)

X_train = clean_train_df[model_features].copy()
y_train = clean_train_df["LogClosePrice"]

X_val = clean_validation_df[model_features].copy()
y_val = clean_validation_df["LogClosePrice"]

# Prepare the April 2026 test dataset
X_test = clean_test_df[model_features].copy()
y_test = clean_test_df["LogClosePrice"]

In [28]:
# Calculate Mean Absolute Percentage Error
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Calculate Median Absolute Percentage Error
def mdape(y_true, y_pred):
    return np.median(np.abs((y_true - y_pred) / y_true)) * 100

In [29]:
# Custom scorer (calculate R² on the original ClosePrice scale)
def close_price_r2(estimator, X, y):
    predictions_log = estimator.predict(X)

    actual_prices = np.expm1(y)
    predicted_prices = np.expm1(predictions_log)

    return r2_score(actual_prices, predicted_prices)

def dollar_mape(estimator, X, y):
    return mape(np.expm1(y), np.expm1(estimator.predict(X)))

def dollar_mdape(estimator, X, y):
    return mdape(np.expm1(y), np.expm1(estimator.predict(X)))

## Additional Feature Engineering

This section introduces additional features to capture property characteristics and geographicinformation that may improve home sale price predictions. Model performance will be compared against the original baseline models.

### Bedroom to Bathroom Ratio
Create a `BedBathRatio` feature to capture the relationship between the number of bedrooms and bathrooms in a property.

In [30]:
# Create copies to preserve the original datasets
train_fe = clean_train_df.copy()
validation_fe = clean_validation_df.copy()
test_fe = clean_test_df.copy()

# Add bedroom to bathroom ratio
train_fe = add_bed_bath_ratio(train_fe)
validation_fe = add_bed_bath_ratio(validation_fe)
test_fe = add_bed_bath_ratio(test_fe)

### School District Features

This section incorporates elementary and high school district information using California school district boundary data. Geographic features are assigned to properties through spatial joins, with unified school districts included in both categories.

The same school district features developed in the baseline modeling notebook are reused for XGBoost and LightGBM to ensure a consistent model comparison.

In [31]:
# Load California school district boundaries
districts_df = gpd.read_file("../external_data/GeoJSON.geojson")

# Convert property coords into geographic points
train_geo = gpd.GeoDataFrame(train_fe.copy(), geometry=gpd.points_from_xy(
    train_fe["Longitude"], train_fe["Latitude"]), crs="EPSG:4326")

validation_geo = gpd.GeoDataFrame(validation_fe.copy(),
    geometry=gpd.points_from_xy(validation_fe["Longitude"], validation_fe["Latitude"]), crs="EPSG:4326")

test_geo = gpd.GeoDataFrame(test_fe.copy(),
    geometry=gpd.points_from_xy(test_fe["Longitude"], test_fe["Latitude"]), crs="EPSG:4326")

In [32]:
# Elementary school districts (including unified districts)
elementary_districts = districts_df[districts_df["DistrictType"].
     isin(["Elementary", "Unified"])][["DistrictName", "DistrictType", "geometry"]].copy()

# High school districts (including unified districts)
high_school_districts = districts_df[districts_df["DistrictType"].
    isin(["High", "Unified"])][["DistrictName", "DistrictType", "geometry"]].copy()

# Rename columns to match our new features
elementary_districts = elementary_districts.rename(
    columns={"DistrictName": "ElementarySchoolDistrict"})

high_school_districts = high_school_districts.rename(
    columns={"DistrictName": "HighSchoolDistrict"})

In [33]:
# Add school district features to training data
train_fe = add_school_district_features(train_fe, elementary_districts,high_school_districts)

# Add school district features to validation data
validation_fe = add_school_district_features(validation_fe, elementary_districts,high_school_districts)

# Add school district features to test data
test_fe = add_school_district_features(test_fe, elementary_districts, high_school_districts)

### School District Feature Engineering Summary

School district features were assigned to the training, validation, and test datasets using spatial joins. Elementary and high school districts were identified separately, with unified districts included in both categories.

In [34]:
# Impute missing bedroom to bathroom ratios using median and then standardize
ratio_pipeline = Pipeline([("imputation", SimpleImputer(strategy="median")),
                           ("scaling", StandardScaler())])
ratio_column = ["BedBathRatio"]
school_district_columns = ["ElementarySchoolDistrict", "HighSchoolDistrict"]

# Target encoding for school district features
school_district_pipeline = Pipeline([("target_encoding", TargetEncoder(target_type="continuous", cv=5, shuffle=True, random_state=42))])

### Feature-Engineered Preprocessing

Two preprocessing configurations are created to evaluate the contribution of the engineered features:
* Bedroom to Bathroom Ratio: Extends the baseline preprocessor with the BedBathRatio feature, using median imputation and standardization.
* Bedroom to Bathroom Ratio + School Districts: Adds school district features to the ratio-based preprocessor using target encoding with cross-fitting to reduce target leakage.

In [35]:
# Extend baseline preprocessing with bedroom to bathroom ratio
preprocessor_ratio = ColumnTransformer([
    ("area", area_pipeline, area_columns),
    ("property", property_pipeline, property_columns),
    ("coordinates", coordinate_pipeline, coordinate_columns),
    ("property_age", property_age_pipeline, age_columns),
    ("binary", binary_pipeline, binary_columns),
    ("binary_missing", "passthrough", binary_missing_columns),
    ("categorical", categorical_pipeline, categorical_columns),

    # New feature
    ("bed_bath_ratio", ratio_pipeline, ratio_column)])

In [36]:
# Extend baseline preprocessing with both engineered features
preprocessor_fe = ColumnTransformer([
    ("area", area_pipeline, area_columns),
    ("property", property_pipeline, property_columns),
    ("coordinates", coordinate_pipeline, coordinate_columns),
    ("property_age", property_age_pipeline, age_columns),
    ("binary", binary_pipeline, binary_columns),
    ("binary_missing", "passthrough", binary_missing_columns),
    ("categorical", categorical_pipeline, categorical_columns),
    # New features
    ("bed_bath_ratio", ratio_pipeline, ratio_column),
    ("school_district", school_district_pipeline, school_district_columns)])

In [37]:
# Additional engineered features
ratio_features = ["BedBathRatio"]
new_feature_columns = ["BedBathRatio", "ElementarySchoolDistrict", "HighSchoolDistrict"]

# Create copies of the original modeling datasets
X_train_ratio = X_train.copy()
X_validation_ratio = X_val.copy()
X_test_ratio = X_test.copy()

X_train_fe = X_train.copy()
X_validation_fe = X_val.copy()
X_test_fe = X_test.copy()

# Add bedroom to bathroom ratio
for X, df in [(X_train_ratio, train_fe), (X_validation_ratio, validation_fe), (X_test_ratio, test_fe)]:
    X["BedBathRatio"] = df["BedBathRatio"]

# Add all engineered features
for X, df in [(X_train_fe, train_fe), (X_validation_fe, validation_fe), (X_test_fe, test_fe)]:
    for col in new_feature_columns:
        X[col] = df[col]

# Define model feature lists
model_features_ratio = model_features + ratio_features
model_features_fe = model_features + new_feature_columns

### Month-Based Cross-Validation

Use expanding-window cross-validation to evaluate the XGBoost and LightGBM models on chronological monthly splits.

Each fold trains on all preceding months and validates on the following month. The entire preprocessing pipeline is fitted independently within each training fold to prevent data leakage.

In [38]:
# Sort training records chronologically by source month
train_sorted = clean_train_df.sort_values("SourceMonth", kind="stable")

# Identify available training months
months = sorted(train_sorted["SourceMonth"].unique())

# Use the last five months for validation
validation_months = months[-5:]

cv_splits = []

for validation_month in validation_months:

    train_idx = np.where(train_sorted["SourceMonth"] < validation_month)[0]
    val_idx = np.where(train_sorted["SourceMonth"] == validation_month)[0]

    cv_splits.append((train_idx, val_idx))

### Advanced Model Development

XGBoost and LightGBM models are developed using three feature configurations: original features, bedroom-to-bathroom ratio, and bedroom-to-bathroom ratio combined with school district information.

The same preprocessing steps and chronological training data are used across all configurations to ensure a consistent comparison of model performance.

In [46]:
# XGBoost with bedroom to bathroom ratio and school district
xgboost_pipeline = Pipeline([("coordinate_imputation", CoordinateImputer()),
                             ("preprocessing", preprocessor),
                             ("model", XGBRegressor(random_state=42))])

# XGBoost with bedroom to bathroom ratio
xgboost_pipeline_ratio = Pipeline([("coordinate_imputation", CoordinateImputer()),
                                   ("preprocessing", preprocessor_ratio),
                                   ("model", XGBRegressor(random_state=42))])
                                     
# XGBoost with both bathroom to bedroom ratio and school district 
xgboost_pipeline_fe = Pipeline([("coordinate_imputation", CoordinateImputer()),
                                ("preprocessing", preprocessor_fe),
                                ("model", XGBRegressor(random_state=42))])

In [47]:
 # LightGBM with original features
lightgbm_pipeline = Pipeline([("coordinate_imputation", CoordinateImputer()),
                              ("preprocessing", preprocessor),
                              ("model", LGBMRegressor(random_state=42, force_row_wise=True, verbosity=-1))])

# LightGBM with bedroom to bathroom ratio
lightgbm_pipeline_ratio = Pipeline([("coordinate_imputation", CoordinateImputer()),
                                    ("preprocessing", preprocessor_ratio),
                                    ("model", LGBMRegressor(random_state=42, force_row_wise=True, verbosity=-1))])

# LightGBM with bedroom to bathroom ratio and school districts
lightgbm_pipeline_fe = Pipeline([("coordinate_imputation", CoordinateImputer()),
                                 ("preprocessing", preprocessor_fe),
                                 ("model", LGBMRegressor(random_state=42, n_jobs=-1, force_row_wise=True, verbosity=-1))])

In [48]:
# Sort feature-engineered training data using the same order as baseline
train_sorted_fe = train_fe.sort_values("SourceMonth").reset_index(drop=True)

# Prepare features for cross-validation
X_cv = train_sorted_fe[model_features].copy()
X_cv_ratio = train_sorted_fe[model_features_ratio].copy()
X_cv_fe = train_sorted_fe[model_features_fe].copy()

# Target variable
y_cv_fe = train_sorted_fe["LogClosePrice"].copy()

### Month-Based Cross-Validation Results
XGBoost and LightGBM are evaluated using expanding-window cross-validation across five consecutive validation months. Each model is evaluated using three feature configurations: original features, bedroom-to-bathroom ratio, and bedroom-to-bathroom ratio combined with school district information. Performance is measured using R², MAPE, and MdAPE on the original closing-price scale.

In [50]:
# Evaluate advanced models using month-based cross-validation
cv_results_advanced = []

for name, pipeline, X in [
    ("Default XGBoost", xgboost_pipeline, X_cv),
    ("XGBoost + Ratio", xgboost_pipeline_ratio, X_cv_ratio),
    ("XGBoost + Ratio + School Districts", xgboost_pipeline_fe, X_cv_fe),

    ("Default LightGBM", lightgbm_pipeline, X_cv),
    ("LightGBM + Ratio", lightgbm_pipeline_ratio, X_cv_ratio),
    ("LightGBM + Ratio + School Districts", lightgbm_pipeline_fe, X_cv_fe)]:

    # Evaluate using chronological cross-validation
    scores = cross_validate(pipeline, X, y_cv_fe, cv=cv_splits, scoring={"R2": close_price_r2, "MAPE": dollar_mape, "MdAPE": dollar_mdape})

    # Store mean cross-validation metrics
    cv_results_advanced.append({"Model": name, "Mean CV R²": scores["test_R2"].mean(),
                                "Mean CV MAPE (%)": scores["test_MAPE"].mean(),
                                "Mean CV MdAPE (%)": scores["test_MdAPE"].mean()})

# Create results DataFrame
cv_results_advanced_df = pd.DataFrame(cv_results_advanced)

# Display results
display(cv_results_advanced_df.round(4))

,Model,Mean CV R²,Mean CV MAPE (%),Mean CV MdAPE (%)
0,Default XGBoost,0.8843,12.1048,8.2925
1,XGBoost + Ratio,0.8865,12.0999,8.3229
2,XGBoost + Ratio + School Districts,0.8864,12.0317,8.2596
3,Default LightGBM,0.8699,12.8442,8.8694
4,LightGBM + Ratio,0.8692,12.8657,8.9085
5,LightGBM + Ratio + School Districts,0.8710,12.7783,8.8455


### Hyperparameter Tuning (XGBoost)

Based on the initial month-based cross-validation results, XGBoost with the bedroom to bathroom ratio and school district features was selected for hyperparameter tuning, as it achieved the highest mean CV R² among the six advanced model configurations evaluated.

Hyperparameter tuning was performed using RandomizedSearchCV to improve model performance by exploring different combinations of tree depth (max_depth), learning rate (learning_rate), and number of estimators (n_estimators). The search evaluated eight randomly selected hyperparameter combinations using the existing five-fold, month-based cross-validation strategy. Model performance was evaluated using R² on the original closing-price scale.

The best hyperparameter configuration was selected based on the highest mean cross-validation R² and automatically refitted on the full training dataset using refit=True.

In [51]:
# Define hyperparameter values to explore
param_distributions = {"model__max_depth": [3, 5, 7], "model__learning_rate": [0.03, 0.05, 0.1], "model__n_estimators": [100, 300, 500]}

# Configure randomized hyperparameter search for XGBoost
xgboost_search = RandomizedSearchCV(estimator=xgboost_pipeline_fe, param_distributions=param_distributions, n_iter=8,
                                    scoring=close_price_r2, cv=cv_splits, random_state=42, refit=True)

# Find the best hyperparameters using chronological cross-validation
xgboost_search.fit(X_cv_fe, y_cv_fe)

# Print the best hyperparameter combination and its mean CV R²
print("Best parameters:", xgboost_search.best_params_)
print("Best CV R²:", xgboost_search.best_score_)

Best parameters: {'model__n_estimators': 500, 'model__max_depth': 7, 'model__learning_rate': 0.05}
Best CV R²: 0.892776500376047


In [52]:
# Retrieve the tuned XGBoost pipeline
best_xgboost = xgboost_search.best_estimator_

# Define all initial and tuned models
models = [("XGBoost", xgboost_pipeline, X_cv, X_val, False), ("XGBoost + Ratio", xgboost_pipeline_ratio,X_cv_ratio, X_validation_ratio, False),
          ("XGBoost + Ratio + School Districts", xgboost_pipeline_fe, X_cv_fe, X_validation_fe, False),
          ("LightGBM", lightgbm_pipeline, X_cv, X_val, False), ("LightGBM + Ratio", lightgbm_pipeline_ratio, X_cv_ratio, X_validation_ratio, False),
          ("LightGBM + Ratio + School Districts", lightgbm_pipeline_fe, X_cv_fe, X_validation_fe, False),
          ("XGBoost + Ratio + School Districts (Tuned-> n_estimators': 500, max_depth: 7, learning_rate': 0.05)", best_xgboost, X_cv_fe, X_validation_fe, True)]

# Store training and validation results
validation_results_advanced = []

# Convert actual prices back to dollars
y_actual_train = np.expm1(y_cv_fe)
y_actual_val = np.expm1(y_val)

for name, pipeline, X_train_model, X_val_model, already_fitted in models:
    # Fit initial models on the complete training dataset((Tuned XGBoost was already fitted during RandomizedSearchCV))
    if not already_fitted:
        pipeline.fit(X_train_model, y_cv_fe)

    # Predict training and validation log closing prices
    y_pred_train_log = pipeline.predict(X_train_model)
    y_pred_val_log = pipeline.predict(X_val_model)

    # Convert predictions back to dollars
    y_pred_train = np.expm1(y_pred_train_log)
    y_pred_val = np.expm1(y_pred_val_log)

    # Calculate training R² and validation metrics
    validation_results_advanced.append({"Model": name, "Train R²": r2_score(y_actual_train, y_pred_train), "Validation R²": r2_score(y_actual_val, y_pred_val),
        "Validation MAPE (%)": mape(y_actual_val, y_pred_val), "Validation MdAPE (%)": mdape(y_actual_val, y_pred_val)})

# Create comparison table
validation_results_advanced_df = pd.DataFrame(validation_results_advanced)

# Display results
display(validation_results_advanced_df.round(4))

,Model,Train R²,Validation R²,Validation MAPE (%),Validation MdAPE (%)
0,XGBoost,0.9116,0.8922,11.7885,8.3313
1,XGBoost + Ratio,0.9114,0.8932,11.7381,8.3141
2,XGBoost + Ratio + School Districts,0.9109,0.8941,11.6726,8.3155
3,LightGBM,0.8803,0.8749,12.5560,9.0333
4,LightGBM + Ratio,0.8811,0.8746,12.5531,9.0049
5,LightGBM + Ratio + School Districts,0.8814,0.8740,12.5245,8.9319
6,XGBoost + Ratio + School Districts (Tuned-> n_...,0.9232,0.9007,11.2211,7.9167


### Comparison with Random Forest Baseline

* The best-performing Random Forest model from the previous notebook (04_model_comparision.ipynb), which includes the bedroom to bathroom ratio and school district features, is included as a baseline for comparison with XGBoost and LightGBM.

* The Random Forest model achieved a training R² of `0.9640` and a validation R² of `0.8845`, with a validation MAPE of `11.3881%` and MdAPE of `7.6614%`.

* The comparison evaluates whether the advanced models, including the tuned XGBoost model, improve prediction accuracy and generalization performance on the March 2026 validation dataset.

* The tuned XGBoost model with the bedroom to bathroom ratio and school district features achieved the highest validation R² of `0.9007`, with a training R² of `0.9232`, validation MAPE of `11.2211%`, and MdAPE of `7.9167%`.

* Compared with Random Forest, the tuned XGBoost model improved validation R² from `0.8845` to `0.9007` and slightly reduced MAPE from `11.3881%` to `11.2211%`. However, Random Forest maintained a lower MdAPE (`7.6614%` versus `7.9167%`).

* The smaller gap between training and validation R² for tuned XGBoost also suggests better generalization and less overfitting compared with Random Forest.

In [53]:
# Random Forest baseline results from the previous notebook(04_model_comparision.ipynb)
rf_baseline = pd.DataFrame([{"Model": "Random Forest + Ratio + School Districts", "Train R²": 0.9640,
                             "Validation R²": 0.8845, "Validation MAPE (%)": 11.3881, "Validation MdAPE (%)": 7.6614}])

# Combine Random Forest baseline with advanced models
validation_comparison_df = pd.concat([rf_baseline, validation_results_advanced_df], ignore_index=True)

# Sort models by validation R²
validation_comparison_df = validation_comparison_df.sort_values("Validation R²", ascending=False)

display(validation_comparison_df.round(4))

,Model,Train R²,Validation R²,Validation MAPE (%),Validation MdAPE (%)
7,XGBoost + Ratio + School Districts (Tuned-> n_...,0.9232,0.9007,11.2211,7.9167
3,XGBoost + Ratio + School Districts,0.9109,0.8941,11.6726,8.3155
2,XGBoost + Ratio,0.9114,0.8932,11.7381,8.3141
1,XGBoost,0.9116,0.8922,11.7885,8.3313
0,Random Forest + Ratio + School Districts,0.9640,0.8845,11.3881,7.6614
4,LightGBM,0.8803,0.8749,12.5560,9.0333
5,LightGBM + Ratio,0.8811,0.8746,12.5531,9.0049
6,LightGBM + Ratio + School Districts,0.8814,0.8740,12.5245,8.9319


### Next Steps

The following tasks will be completed after the team reviews and finalizes the feature engineering approach:

* Feature Engineering: Incorporate any agreed-upon feature additions or modifications.
* Model Evaluation: Reevaluate the models if the feature set changes and select the best-performing model.
* Test Evaluation: Evaluate the selected model on the held-out April 2026 test dataset.
* Further Analysis: Conduct residual analysis, error analysis across different price ranges, and additional visualizations to better understand model performance and limitations.

Note: Test evaluation and further analysis are intentionally deferred until the team agrees on the final feature set to ensure a consistent and fair comparison.